In [13]:
import spacy
from spacy.matcher import PhraseMatcher
import json
import pickle
import PyPDF2

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

def extract_skills(cv_text):
    with open("courses.json", "r", encoding="utf-8") as f:
        courses_data = json.load(f)
    skills_list = list(courses_data.keys())
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns = [nlp(skill) for skill in skills_list]
    matcher.add("SKILLS", patterns)
    doc = nlp(cv_text)
    matches = matcher(doc)
    synonyms = {"EDR": "endpoint security", "endpoint security": "EDR"}
    found_skills = set([doc[start:end].text for _, start, end in matches])
    for s in list(found_skills):
        if s in synonyms:
            found_skills.add(synonyms[s])
    return list(found_skills)

def train_skill_model(data):
    descriptions = [job["description"] for job in data]
    skills = [job["skills"] for job in data]
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(skills)
    tfidf = TfidfVectorizer(max_features=3000)
    X = tfidf.fit_transform(descriptions)
    model = MultiOutputClassifier(RandomForestClassifier())
    model.fit(X, y)
    pickle.dump(model, open("skill_model.pkl", "wb"))
    pickle.dump(tfidf, open("tfidf.pkl", "wb"))
    pickle.dump(mlb, open("mlb.pkl", "wb"))

def predict_required_skills(job_description):
    model = pickle.load(open("skill_model.pkl", "rb"))
    tfidf = pickle.load(open("tfidf.pkl", "rb"))
    mlb = pickle.load(open("mlb.pkl", "rb"))
    X = tfidf.transform([job_description])
    y_pred = model.predict(X)
    return list(mlb.inverse_transform(y_pred)[0])

def calculate_semantic_score(cv_text, job_description):
    emb = sbert_model.encode([cv_text, job_description])
    score = cosine_similarity([emb[0]], [emb[1]])[0][0]
    return round(max(0, float(score) * 100), 2)

def calculate_skill_match(cv_skills, required_skills):
    if not required_skills:
        return 0
    matched = len(set(cv_skills) & set(required_skills))
    return round((matched / len(required_skills)) * 100, 2)

def calculate_final_score(cv_text, job_desc, cv_skills, required_skills):
    semantic = calculate_semantic_score(cv_text, job_desc)
    skill_match = calculate_skill_match(cv_skills, required_skills)
    final = 0.3 * semantic + 0.7 * skill_match
    return round(final, 2), semantic, skill_match

def get_missing_skills(cv_skills, required_skills):
    return [s for s in required_skills if s not in cv_skills]

def get_course_recommendations(missing_skills):
    with open("courses.json", "r", encoding="utf-8") as f:
        courses = json.load(f)
    result = []
    for skill in missing_skills:
        if skill in courses:
            result.append({
                "skill": skill,
                "course_name": courses[skill]["name"],
                "platform": courses[skill]["platform"],
                "link": courses[skill]["link"]
            })
    return result

def course_impact_analysis(current_score, missing_skills):
    if not missing_skills:
        return []
    core_skills = {"Python", "Linux", "Windows", "SOC", "SIEM", "EDR", "NGFW", "penetration testing"}
    impact_result = []
    temp = current_score
    for skill in missing_skills:
        impact = 7 if skill in core_skills else 4
        temp = min(temp + impact, 100)
        impact_result.append({
            "skill": skill,
            "after_learning_score": round(temp, 1),
            "increase": impact
        })
    return impact_result

def extract_text_from_pdf(pdf_path):
    text = ""
    with open(pdf_path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

def analyze_cv(cv_pdf_path, jobs_list):
    cv_text = extract_text_from_pdf(cv_pdf_path)
    cv_skills = extract_skills(cv_text)
    results = []
    for job in jobs_list:
        desc = job["description"]
        required = predict_required_skills(desc)
        final_score, semantic, skill_match = calculate_final_score(cv_text, desc, cv_skills, required)
        missing = get_missing_skills(cv_skills, required)
        courses = get_course_recommendations(missing)
        impact = course_impact_analysis(final_score, missing)
        results.append({
            "job_title": job["title"],
            "scores": {
                "final": final_score,
                "skill_match": skill_match,
                "semantic": semantic
            },
            "cv_skills": cv_skills,
            "required_skills": required,
            "missing_skills": missing,
            "courses": courses,
            "learning_path": impact
        })
    return sorted(results, key=lambda x: x["scores"]["final"], reverse=True)


with open("busy_az_cybersecurity_dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

jobs = [
    {
        "title": item["title_normalized_en"],
        "description": item["description"],
        "skills": item["skills"]
    }
    for item in dataset
]

train_skill_model(jobs)

results = analyze_cv("Anar Firudinov.pdf", jobs)

print(json.dumps(results[:3], ensure_ascii=False, indent=2))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 8044.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[
  {
    "job_title": "Cybersecurity Engineer",
    "scores": {
      "final": 37.62,
      "skill_match": 33.33,
      "semantic": 47.63
    },
    "cv_skills": [
      "SIEM",
      "LINUX",
      "WAF",
      "Linux",
      "Owasp",
      "English",
      "Python"
    ],
    "required_skills": [
      "English",
      "Linux",
      "NGFW",
      "Russian",
      "SIEM",
      "SOAR",
      "WAF",
      "Windows",
      "endpoint security",
      "pre-sales",
      "threat intelligence",
      "vulnerability assessment"
    ],
    "missing_skills": [
      "NGFW",
      "Russian",
      "SOAR",
      "Windows",
      "endpoint security",
      "pre-sales",
      "threat intelligence",
      "vulnerability assessment"
    ],
    "courses": [
      {
        "skill": "NGFW",
        "course_name": "Next Generation Firewalls",
        "platform": "Pluralsight",
        "link": "https://www.pluralsight.com/courses/next-gen-firewalls"
      },
      {
        "skill": "Russian",
       